# 02 Create Frozen Splits and Manifests

This notebook creates the deterministic train/validation/test manifest that all downstream notebooks consume. Thresholds are selected only on the validation split; final metrics are computed only on the test split.


In [11]:
from pathlib import Path
import random
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [7]:
SEED = 42
REPO_ROOT = Path('../..').resolve()
DATA_ROOT = REPO_ROOT / 'data' / '03_broach_dataset'
MANIFEST_DIR = REPO_ROOT / 'reports' / 'manifests'
MANIFEST_PATH = MANIFEST_DIR / 'broach_dataset_split_seed42.csv'
SUMMARY_PATH = MANIFEST_DIR / 'broach_split_summary.csv'

VALIDATION_TEST_FRACTION = 0.5
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"

TEST_LABELS_CSV  = TEST_DIR / "labels.csv"
VALIDATION_TEST_FRACTION = 0.5 # split for test and validation sets


In [8]:
def list_train_files(train_dir: Path, extensions):
    files = []
    for ext in extensions:
        files.extend([p for p in train_dir.iterdir() if p.is_file() and p.suffix == ext])
    files = sorted(set(files))
    return files

In [9]:
# Read test data
labels_df = pd.read_csv(TEST_LABELS_CSV)

# Required columns check
required_cols = {"filename", "label"}
missing = required_cols - set(labels_df.columns)
if missing:
    raise RuntimeError(f"labels.csv missing columns: {missing}. Available: {list(labels_df.columns)}")

# Types
labels_df["label"] = labels_df["label"].astype(int)

In [10]:
# Read Train-Data (only label=0)
test_extensions = sorted({Path(fn).suffix for fn in labels_df["filename"].astype(str)})
if not test_extensions:
    raise RuntimeError("Konnte aus labels.csv keine Dateiendung ableiten.")

train_files = list_train_files(TRAIN_DIR, test_extensions)
if not train_files:
    raise RuntimeError(f"Keine Dateien im train-Ordner gefunden mit Extensions: {test_extensions}")

train_rows = [{
    "filename": p.name,  
    "label": 0,
    "split": "train"
} for p in train_files]

In [ ]:
# Test-Data: Split in test/validation
test_filenames = labels_df["filename"].astype(str).tolist()
test_labels = labels_df["label"].tolist()

f_test = VALIDATION_TEST_FRACTION 
X_val, X_test, y_val, y_test = train_test_split(
    test_filenames, test_labels,
    test_size=f_test,
    stratify=test_labels,
    random_state=SEED
)

val_rows = [{
    "filename": fn,
    "label": int(lab),
    "split": "validation"
} for fn, lab in zip(X_val, y_val)]

test_rows = [{
    "filename": fn,
    "label": int(lab),
    "split": "test"
} for fn, lab in zip(X_test, y_test)]

In [14]:
# create and write manifest
manifest = pd.DataFrame(train_rows + val_rows + test_rows)

# Optional: Checks
if manifest["filename"].duplicated().any():
    print("Hinweis: Es gibt doppelte filename-Einträge im Manifest (z.B. train/test überlappen).")

# Sicherstellen, dass Spaltenreihenfolge stimmt
manifest = manifest[["filename", "label", "split"]].sort_values(["split", "label", "filename"])

summary = (manifest
           .groupby(["split", "label"])
           .size()
           .rename("n")
           .reset_index())

summary.to_csv(SUMMARY_PATH, index=False)
manifest.to_csv(MANIFEST_PATH, index=False)
print(f"Wrote {MANIFEST_PATH}")
print(manifest["split"].value_counts())
print(summary)

Wrote /home/sebastian/Dokumente/01_VSC/spectrogram-anomaly-ae/reports/manifests/broach_dataset_split_seed42.csv
split
train         5000
test          2500
validation    2500
Name: count, dtype: int64
        split  label     n
0        test      0  2478
1        test      1    22
2       train      0  5000
3  validation      0  2477
4  validation      1    23
